In [3]:
pip install torch_geometric

  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached frozenlist-1.8.0-cp311-cp311-win_amd64.whl.metadata (21 kB)
  Using cached propcache-0.4.1-cp311-cp311-win_amd64.whl.metadata (14 kB)
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 1.3/1.3 MB 8.1 MB/s  0:00:00
Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl (15 kB)
Using cached aiosignal-1.4.0-py3-none-any.whl (7.5 kB)
Using cached frozenlist-1.8.0-cp311-cp311-win_amd64.whl (44 kB)
Using cached propcache-0.4.1-cp311-cp311-win_amd64.whl (41 kB)

   ------------- -------------------------- 3/9 [frozenlist]
   ---------------------- ----------------- 5/9 [yarl]
   ------------------------------- -------- 7/9 [aiohttp]
   ------------------------------- -------- 7/9 [aiohttp]
   ------------------------------- -------- 7/9 [aiohttp]
   ------------------------------- -

In [ ]:
import torch
print(torch.version.cuda)
print(torch.cuda.is_available())

12.1
True


In [ ]:
"""
Đọc database → Build graph → Save
Chạy 1 lần: python build_graph.py
"""
import torch
from torch_geometric.data import HeteroData
from sqlalchemy import create_engine
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

# 1. Đọc dữ liệu từ database
engine = create_engine('sqlite:///data/job_rec.db')

df_users = pd.read_sql("SELECT * FROM users", engine)
df_jobs = pd.read_sql("SELECT * FROM jobs", engine)
df_skills = pd.read_sql("SELECT * FROM skills", engine)
df_applications = pd.read_sql("SELECT * FROM applications", engine)
df_user_skills = pd.read_sql("SELECT * FROM user_skills", engine)
df_job_skills = pd.read_sql("SELECT * FROM job_skills", engine)

print(f"Users: {len(df_users)}, Jobs: {len(df_jobs)}, Skills: {len(df_skills)}")

# 2. Tạo mapping ID → index
user_map = {uid: i for i, uid in enumerate(df_users['user_id'])}
job_map = {jid: i for i, jid in enumerate(df_jobs['job_id'])}
skill_map = {sid: i for i, sid in enumerate(df_skills['skill_id'])}

# 3. Tạo features (đơn giản)
graph = HeteroData()

# User features: [location, education, experience]
le_loc = LabelEncoder()
user_feats = np.column_stack([
    le_loc.fit_transform(df_users['location']),
    df_users['education'].map({'Cao đẳng': 1, 'Đại học': 2, 'Thạc sĩ': 3, 'Tiến sĩ': 4}),
    df_users['years_experience']
])
graph['user'].x = torch.FloatTensor(user_feats)

# Job features: [location, salary_min, salary_max]
job_feats = np.column_stack([
    le_loc.transform(df_jobs['location']),
    df_jobs['salary_min'].fillna(0),
    df_jobs['salary_max'].fillna(0)
])
graph['job'].x = torch.FloatTensor(job_feats)

# Skill features: one-hot
graph['skill'].x = torch.eye(len(df_skills))

# 4. Tạo edges
def make_edge_index(df, src_col, dst_col, src_map, dst_map):
    src = df[src_col].map(src_map).dropna().astype(int).values
    dst = df[dst_col].map(dst_map).dropna().astype(int).values
    return torch.LongTensor([src, dst])

graph['user', 'applies', 'job'].edge_index = make_edge_index(
    df_applications, 'user_id', 'job_id', user_map, job_map
)

graph['user', 'has', 'skill'].edge_index = make_edge_index(
    df_user_skills, 'user_id', 'skill_id', user_map, skill_map
)

graph['job', 'requires', 'skill'].edge_index = make_edge_index(
    df_job_skills, 'job_id', 'skill_id', job_map, skill_map
)

# 5. Train/Val/Test split
from torch_geometric.transforms import RandomLinkSplit

transform = RandomLinkSplit(
    num_val=0.1,
    num_test=0.1,
    edge_types=[('user', 'applies', 'job')],
    neg_sampling_ratio=1.0
)

train_data, val_data, test_data = transform(graph)

# 6. Save
torch.save(train_data, 'data/graphs/train_graph.pt')
torch.save(val_data, 'data/graphs/val_graph.pt')
torch.save(test_data, 'data/graphs/test_graph.pt')

print("✅ Graph saved!")
print(f"Train edges: {train_data['user', 'applies', 'job'].edge_label_index.shape[1]}")

In [ ]:
"""
GraphSAGE model - ĐƠN GIẢN NHẤT
"""
import torch
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv, to_hetero

class JobRecModel(torch.nn.Module):
    def __init__(self, hidden_dim=64):
        super().__init__()
        
        # 2 layers GraphSAGE
        self.conv1 = SAGEConv((-1, -1), hidden_dim)
        self.conv2 = SAGEConv((-1, -1), hidden_dim)
    
    def forward(self, x, edge_index):
        # Layer 1
        x = self.conv1(x, edge_index)
        x = x.relu()
        x = F.dropout(x, p=0.5, training=self.training)
        
        # Layer 2
        x = self.conv2(x, edge_index)
        return x


# Hàm tính loss
def compute_loss(model, data, edge_type):
    # Forward pass
    z = model(data.x_dict, data.edge_index_dict)
    
    # Get embeddings
    src_type, _, dst_type = edge_type
    edge_label_index = data[edge_type].edge_label_index
    edge_label = data[edge_type].edge_label
    
    # Predict edge scores
    src_emb = z[src_type][edge_label_index[0]]
    dst_emb = z[dst_type][edge_label_index[1]]
    pred = (src_emb * dst_emb).sum(dim=-1)
    
    # Binary cross-entropy
    loss = F.binary_cross_entropy_with_logits(pred, edge_label)
    return loss


# Hàm đánh giá
@torch.no_grad()
def evaluate(model, data, edge_type):
    model.eval()
    z = model(data.x_dict, data.edge_index_dict)
    
    src_type, _, dst_type = edge_type
    edge_label_index = data[edge_type].edge_label_index
    edge_label = data[edge_type].edge_label
    
    src_emb = z[src_type][edge_label_index[0]]
    dst_emb = z[dst_type][edge_label_index[1]]
    pred = (src_emb * dst_emb).sum(dim=-1).sigmoid()
    
    # AUC
    from sklearn.metrics import roc_auc_score
    auc = roc_auc_score(edge_label.cpu(), pred.cpu())
    return auc

c:\Users\voquy\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NameError: name 'graph' is not defined